In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta

from enum import Enum, IntEnum, StrEnum
from dataclasses import dataclass, field

import json, requests, asyncio
from hyperliquid.info import Info
from hyperliquid.utils import constants

In [2]:
# %pip install hyperliquid-python-sdk

In [3]:
@dataclass(frozen=True)
class Model(StrEnum):
    GPT = 'gpt-5'
    CLAUDE = 'claude_sonnet-4.5'
    DEEPSEEK = 'deepseek_chat-v3.1'
    GEMINI = 'gemini-2.5-pro'
    GROK = 'grok-4'
    QWEN = 'qwen3-30b'

MODEL_WALLET_ADDRESSES = {
    Model.GPT.value: '0x67293d914eafb26878534571add81f6bd2d9fe06',
    Model.CLAUDE.value: '0x59fa085d106541a834017b97060bcbbb0aa82869', 
    Model.DEEPSEEK.value: '0xc20ac4dc4188660cbf555448af52694ca62b0734',
    Model.GEMINI.value: '0x1b7a7d099a670256207a30dd0ae13d35f278010f', 
    Model.GROK.value: '0x56d652e62998251b56c8398fb11fcfe464c08f84', 
    Model.QWEN.value: 'nan',
}

In [4]:
class LLMTradingBot:
    
    def __init__(self, model: Model, initial_amount: float=10_000.0):
        self.model = model
        self.initial_amount = initial_amount

    def get_wallet_address(self):
        return MODEL_WALLET_ADDRESSES[self.model.value]

    def get_historical_fill_data(self):
        info = Info()
        fill_data = info.user_fills(self.get_wallet_address())
        fill_data = pd.DataFrame(fill_data)
        fill_data['timestamp'] = [datetime.fromtimestamp(unix_timestamp/1e3) for unix_timestamp in fill_data['time'].astype(int)]
        fill_data.set_index('timestamp', inplace=True)
        fill_data.drop(columns=['time'], inplace=True)
        fill_data['cumPnl'] = np.cumsum(fill_data['closedPnl'].astype(float))
        return fill_data

    def get_historical_pnl(self):
        return self.get_historical_fill_data()['cumPnl'] + self.initial_amount

    def plot_historical_pnl(self):
        fill_data = self.get_historical_fill_data()
        plt.figure(figsize=(10, 5))
        plt.plot((fill_data['cumPnl'] + self.initial_amount) / 1e3, 'k-', label=self.model.value)
        plt.xticks(rotation=45)
        plt.ylabel('$K USD')
        plt.title('Cumulative PnL')
        plt.legend()
        plt.show()